# Download Files From Citesphere Group

In [1]:
import requests
import csv
import time
from alive_progress import alive_bar, config_handler
import ipywidgets as widgets
from jupyter_ui_poll import ui_events
from IPython.display import display

from src.CitesphereConnector import CitesphereConnector
from src.AuthObject import AuthObject

## Specify Properties

In the following properties need to be set before continuing:
- `FOLDER_NAME`: path to the folder in which files should be downloaded, can be relative or absolute. Default value downloads files into a folder "download" located next to this notebook.
- `GROUP_ID`: id of the Zotero group that should be downloaded (can be retrieved from the url of a group in Citesphere).
- `CITESPHERE_API_URL`: API endpoint of Citesphere (should end in `/api`).
- `TOKEN`: Citesphere access token.
- `GILES_ROOT`: Base url of Giles.

In [2]:
GROUP_ID = ""
TOKEN = ""

FOLDER_NAME = "download/"
AUTH_TYPE = "oauth"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
CSV_FILE_NAME = f"citesphere_{GROUP_ID}.csv"

# Root/API URLs
CITESPHERE_API_URL = "https://example.edu/example/api"
GILES_ROOT = "https://example.edu/example"

# Giles Endpoints
UPLOAD_BY_PROGRESS = f"{GILES_ROOT}/api/v2/files/upload/check/"
UPLOAD_ENDPOINT = f"{GILES_ROOT}/api/v2/resources/files/upload/"

## Choose Your File Type
Click the checkboxes to indicate which files types you would like to download and then click 'Submit'.

In [ ]:
file_type_1 = widgets.Checkbox(
    value=False, description="Original File", disabled=False, indent=False
)
file_type_2 = widgets.Checkbox(
    value=False, description="Extracted Text File", disabled=False, indent=False
)
submit_button_1 = widgets.Button(description="Submit", button_style="primary")


def on_button_clicked_1(b):
    selected_options = [c.description for c in [file_type_1, file_type_2] if c.value]
    if selected_options:
        submit_button_1.description = "Submitted"
        submit_button_1.button_style = "success"
    else:
        submit_button_1.description = "Not Submitted"
        submit_button_1.button_style = "danger"
        time.sleep(3)
        submit_button_1.description = "Submit"
        submit_button_1.button_style = "primary"


submit_button_1.on_click(on_button_clicked_1)
display(file_type_1, file_type_2, submit_button_1)

with ui_events() as poll:
    while not submit_button_1.description == "Submitted":
        poll(1)
        time.sleep(0.1)

## Choose Your Download Type
Click the checkboxes to indicate which files you would like to download and then click 'Submit'.

In [ ]:
download_type_1 = widgets.Checkbox(
    value=False, description="Fully-Processed Files", disabled=False, indent=False
)
download_type_2 = widgets.Checkbox(
    value=False, description="Unprocessed Files", disabled=False, indent=False
)
submit_button_2 = widgets.Button(description="Submit", button_style="primary")


def on_button_clicked_2(b):
    selected_options = [
        c.description for c in [download_type_1, download_type_2] if c.value
    ]
    if selected_options:
        submit_button_2.description = "Submitted"
        submit_button_2.button_style = "success"
    else:
        submit_button_2.description = "Not Submitted"
        submit_button_2.button_style = "danger"
        time.sleep(3)
        submit_button_1.description = "Submit"
        submit_button_1.button_style = "primary"


submit_button_2.on_click(on_button_clicked_2)
display(download_type_1, download_type_2, submit_button_2)

with ui_events() as poll:
    while not submit_button_2.description == "Submitted":
        poll(1)
        time.sleep(0.1)

## Instantiate Classes
Create instances of CitesphereConnector and AuthObject to use the API

In [5]:
auth_object = AuthObject(auth_type=AUTH_TYPE, access_token=TOKEN)
connector = CitesphereConnector(CITESPHERE_API_URL, auth_object)

## Functions
The following functions do the main work of downloading files and creating the metadata CSV file.

In [7]:
def get_filename_from_response(response):
    """
    Return file name from download_file response

    Args:
        response (Response): The response of the download_file request

    Returns:
        File name (str)
    """
    content_disposition = response.headers.get("Content-Disposition")
    if content_disposition and "filename=" in content_disposition:
        # Extract the filename value
        filename = content_disposition.split("filename=")[1].strip('"')
        return filename
    return None


def download_file(file_id):
    """
    Use file id to download file from Giles and save it to folder

    Args:
        file_id (str): The id of the Giles file to be downloaded.

    Returns:
        None.
    """
    endpoint = f"{GILES_ROOT}/api/v2/resources/files/{file_id}/content"
    try:
        response = requests.get(endpoint, headers=HEADERS)
        filename = get_filename_from_response(response)
        # if we have a filename, we'll download the file
        # this will override files with the same name in the folder FOLDER_NAME!
        if filename:
            with open(FOLDER_NAME + filename, "wb") as file:
                file.write(response.content)
    except Exception as exc:
        return {"error_message": str(exc)}


def get_original_file_id_from_progress(progress_id):
    """
    Return an original file id using the progress id provided by incomplete uploads.

    Args:
        progress_id (str): The progress id of the in-progress upload.

    Returns:
        File id (str)
    """
    try:
        upload = requests.get(UPLOAD_BY_PROGRESS + progress_id, headers=HEADERS).json()
        # if processing in progress
        if "msg" in upload and "uploadId" in upload and upload["uploadId"]:
            try:
                progress_upload = requests.get(
                    UPLOAD_ENDPOINT + upload["uploadId"], headers=HEADERS
                ).json()
                if (
                    "uploadedFile" in upload
                    and upload["uploadedFile"]
                    and upload["uploadedFile"]["id"]
                ):
                    return upload["uploadedFile"]["id"], upload["uploadedFile"][
                        "filename"
                    ]
                else:
                    print(
                        f"Can't download file for:\nProgress Id: {progress_id}\nUpload Id: {upload['uploadId']}."
                    )
            except Exception:
                print(progress_upload)
        else:
            if (
                "uploadedFile" in upload
                and upload["uploadedFile"]
                and upload["uploadedFile"]["id"]
            ):
                return upload["uploadedFile"]["id"], upload["uploadedFile"]["filename"]
            else:
                print(f"Can't download file for {progress_id}.")
    except Exception as exc:
        return {"error_message": str(exc)}


def get_extracted_id_from_progress(progress_id):
    """
    Return an extracted text file id using the progress id provided by incomplete uploads.

    Args:
        progress_id (str): The progress id of the in-progress upload.

    Returns:
        File id (str)
    """
    try:
        upload = requests.get(UPLOAD_BY_PROGRESS + progress_id, headers=HEADERS).json()
        # if processing in progress
        if "msg" in upload and "uploadId" in upload:
            try:
                progress_upload = requests.get(
                    UPLOAD_ENDPOINT + upload["uploadId"], headers=HEADERS
                ).json()
                if (
                    "extractedText" in progress_upload
                    and progress_upload["extractedText"]
                    and progress_upload["extractedText"]["id"]
                ):
                    return progress_upload["extractedText"]["id"], progress_upload[
                        "extractedText"
                    ]["filename"]
                else:
                    print(f"Can't download file for {progress_id}.")
            except Exception:
                print(progress_upload)
        else:
            if (
                "extractedText" in upload
                and upload["extractedText"]
                and upload["extractedText"]["id"]
            ):
                return upload["extractedText"]["id"], upload["extractedText"][
                    "filename"
                ]
            else:
                print(f"Can't download file for {progress_id}.")
    except Exception as exc:
        return {"error_message": str(exc)}


# Create the CSV file and write the metadata
def write_to_csv(csv_name, item, filename, flag):
    """
    Writes a row of column names and then writes item fields to rows of a CSV file.

    Args:
        csv_name (str): The name of the resulting CSV file where the metadata will populate.
        item (list): A list of item metadata.
        filename (str): The name of the file that was downloaded.
        flag (int):

    Returns:
        None.
    """
    with open(csv_name, "a", newline="") as file:
        writer = csv.writer(file)

        # Check if it's the first time writing to the file
        if flag == 0:
            writer.writerow(
                list(["key", "title", "authors", "editors", "date", "file"])
            )

        writer.writerow(
            list(
                [
                    item["key"],
                    item["title"],
                    item["authors"],
                    item["editors"],
                    item["date"],
                    filename,
                ]
            )
        )

## Download files and extract metadata to CSV
The following script uses the functions above to download files and extract the some of the downloaded file metadata to csv files.

In [ ]:
processed_ids = []
unprocessed_ids = []
page = 0
write_flag = 0

while True:
    page += 1
    # get all potential downloadable items
    items = connector.get_group_items(GROUP_ID, page_number=page)
    if not items["items"]:
        break
    if items and items["items"]:
        for item in items["items"]:
            uploads = item["gilesUploads"]
            if uploads:
                for upload in uploads:
                    try:
                        # if user has selected a downmload type
                        if download_type_1.value or download_type_2.value:
                            if download_type_1.value:
                                # if user has selected a file type
                                if file_type_1.value or file_type_2.value:
                                    if file_type_1.value:
                                        if (
                                            "uploadedFile" in upload
                                            and upload["uploadedFile"]
                                            and upload["uploadedFile"]["id"]
                                        ):
                                            processed_ids.append(
                                                upload["uploadedFile"]["id"]
                                            )
                                            write_to_csv(
                                                CSV_FILE_NAME,
                                                item,
                                                upload["uploadedFile"]["filename"],
                                                write_flag,
                                            )
                                            write_flag = 1
                                    if file_type_2.value:
                                        if (
                                            "extractedText" in upload
                                            and upload["extractedText"]
                                            and upload["extractedText"]["id"]
                                        ):
                                            processed_ids.append(
                                                upload["extractedText"]["id"]
                                            )
                                            write_to_csv(
                                                CSV_FILE_NAME,
                                                item,
                                                upload["extractedText"]["filename"],
                                                write_flag,
                                            )
                                            write_flag = 1
                                else:
                                    raise Exception(
                                        "Please select one or more file types above."
                                    )
                            elif download_type_2.value:
                                if "progressId" in upload and upload["progressId"]:
                                    if file_type_1.value or file_type_2.value:
                                        if file_type_1.value:
                                            if get_original_file_id_from_progress(
                                                upload["progressId"]
                                            ):
                                                id, filename = (
                                                    get_original_file_id_from_progress(
                                                        upload["progressId"]
                                                    )
                                                )
                                                unprocessed_ids.append(id)
                                                write_to_csv(
                                                    CSV_FILE_NAME,
                                                    item,
                                                    filename,
                                                    write_flag,
                                                )
                                                write_flag = 1
                                        if file_type_2.value:
                                            if get_extracted_id_from_progress(
                                                upload["progressId"]
                                            ):
                                                id, filename = (
                                                    get_extracted_id_from_progress(
                                                        upload["progressId"]
                                                    )
                                                )
                                                unprocessed_ids.append(id)
                                                write_to_csv(
                                                    CSV_FILE_NAME,
                                                    item,
                                                    filename,
                                                    write_flag,
                                                )
                                                write_flag = 1
                                    else:
                                        raise Exception(
                                            "Please select one or more file types above."
                                        )
                        else:
                            raise Exception(
                                "Please select one or more download types above."
                            )
                    except Exception:
                        print(f"Encountered an error! {upload}")

config_handler.set_global(max_cols=120, bar="smooth", spinner="waves2", force_tty=True)
# update progress bar during for-loop
with alive_bar(len(processed_ids), title="Fully Processed Files", calibrate=50) as bar:
    for file_id in processed_ids:
        download_file(file_id)
        bar()

with alive_bar(len(unprocessed_ids), title="Unprocessed Files", calibrate=50) as bar:
    for file_id in unprocessed_ids:
        download_file(file_id)
        bar()